In [18]:
import os
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from nltk.tokenize import sent_tokenize
import nltk
nltk.download("punkt")
nltk.download("stopwords")
import re
from sklearn.impute import SimpleImputer
from sentence_transformers import SentenceTransformer
import os
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import shap
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import ast
import nltk
from xgboost import XGBClassifier

[nltk_data] Downloading package punkt to /Users/bao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/bao/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## PREPROCESSING

In [19]:
# Thư mục chứa scripts
folder_path = "./imsdbfilmscripts"

# Danh sách để lưu dữ liệu
data = []

# Duyệt tất cả file .txt trong folder
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        movie_name = os.path.splitext(filename)[0]  # tên file bỏ .txt
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            movie_script = f.read().replace("\n", " ")  # chuyển xuống dòng thành space
        data.append([movie_name, movie_script])

# Tạo DataFrame
df = pd.DataFrame(data, columns=["movie_name", "movie_script"])

# Xuất ra CSV
#df.to_csv("movies_scripts.csv", index=False, encoding="utf-8")
print("Done! CSV saved as movies_scripts.csv")

Done! CSV saved as movies_scripts.csv


In [20]:
df["movie_name"] = df["movie_name"].str.replace("-", " ")

In [21]:
df

,movie_name,movie_script
0,Midnight Express,MIDNIG...
1,Speed,Speed Script at IMSDb. This script was...
2,Big Eyes,...
3,Warrior,WARRIOR ...
4,Hellraiser Hellseeker,HELLRAISER: HELLSEEKER ...
...,...,...
1160,Silver Linings Playbook,SILVER LININGS...
1161,"Prestige, The",THE P...
1162,Four Rooms,"""FOUR R..."
1163,"Theory of Everything, The",THE THEORY...


In [22]:
ratings = pd.read_csv("data/title.ratings.tsv", sep="\t")
basics = pd.read_csv("data/title.basics.tsv", sep="\t", low_memory=False)

# Keep only movies
basics = basics[basics["titleType"] == "movie"]

# Merge ratings with titles
imdb = basics.merge(ratings, on="tconst")

# Keep the most popular version for each exact title
imdb_unique = imdb.sort_values("numVotes", ascending=False).drop_duplicates("primaryTitle")

In [23]:
df = df.merge(
    imdb_unique[["primaryTitle", "averageRating"]],
    left_on="movie_name",
    right_on="primaryTitle",
    how="left"
)

# Remove the extra column
df = df.drop(columns=["primaryTitle"])

In [24]:
df

,movie_name,movie_script,averageRating
0,Midnight Express,MIDNIG...,7.5
1,Speed,Speed Script at IMSDb. This script was...,7.3
2,Big Eyes,...,6.9
3,Warrior,WARRIOR ...,8.1
4,Hellraiser Hellseeker,HELLRAISER: HELLSEEKER ...,NaN
...,...,...,...
1160,Silver Linings Playbook,SILVER LININGS...,7.7
1161,"Prestige, The",THE P...,NaN
1162,Four Rooms,"""FOUR R...",6.7
1163,"Theory of Everything, The",THE THEORY...,NaN


In [25]:
df.dtypes

movie_name           str
movie_script         str
averageRating    float64
dtype: object

In [26]:
df.isna().sum()

movie_name         0
movie_script       0
averageRating    359
dtype: int64

In [27]:
imdb_titles = imdb_unique["primaryTitle"].tolist()

# Fuzzy match function
def fuzzy_match(title, threshold=90):
    match = process.extractOne(title, imdb_titles, scorer=fuzz.ratio)
    if match and match[1] >= threshold:
        # Get rating
        matched_title = match[0]
        rating = imdb_unique.loc[
            imdb_unique["primaryTitle"] == matched_title, "averageRating"
        ].values[0]
        return rating
    return None

# Apply fuzzy matching only for missing ratings
missing_mask = df["averageRating"].isna()
df.loc[missing_mask, "averageRating"] = df.loc[missing_mask, "movie_name"].apply(lambda x: fuzzy_match(str(x)))

In [28]:
df.isna().sum()

movie_name         0
movie_script       0
averageRating    284
dtype: int64

In [29]:
# Drop rows with null ratings
df = df.dropna(subset=["averageRating"])

# Reset index (optional)
df = df.reset_index(drop=True)


In [30]:
df.head()

,movie_name,movie_script,averageRating
0,Midnight Express,MIDNIG...,7.5
1,Speed,Speed Script at IMSDb. This script was...,7.3
2,Big Eyes,...,6.9
3,Warrior,WARRIOR ...,8.1
4,Hannah and Her Sisters,<b><!-- </b> <b>/* </b>Break-out-of-frames ...,7.8


In [ ]:
def preprocess_script(script: str) -> str:
    # Remove INT. / EXT. scene headings (common screenplay format)
    script = re.sub(r"^\s*(INT\.|EXT\.)[^\n]*\n", "", script,
                    flags=re.MULTILINE | re.IGNORECASE)

    # Remove character cue lines: lines that are ALL CAPS (and optional spaces)
    # Typical pattern: "    ANDY" or "JOKER (V.O.)" on its own line
    script = re.sub(r"^\s*[A-Z][A-Z\s\.\(\)\']{2,}\s*$", "",
                    script, flags=re.MULTILINE)

    # Remove parentheticals like (V.O.), (CONT'D), (quietly)
    script = re.sub(r"\(.*?\)", "", script)
    script = re.sub(r'</?b>', '', script)
    script = re.sub(r'<!--.*?-->', '', script, flags=re.DOTALL)
    # Remove C-style comments /* ... */
    script = re.sub(r'/\*.*?\*/', '', script, flags=re.DOTALL)
    # Remove JavaScript single-line comments // ...
    script = re.sub(r'//.*', '', script)
    # Strip leading/trailing whitespace and normalize spaces
    script = re.sub(r'\s+', ' ', script).strip()

    # Remove lines that look like screenplay headers or transitions
    # e.g. "FADE IN:", "CUT TO:", "DISSOLVE TO:"
    script = re.sub(r"^\s*(FADE IN|FADE OUT|CUT TO|DISSOLVE TO|SMASH CUT).*$",
                    "", script, flags=re.MULTILINE | re.IGNORECASE)

    # Collapse multiple blank lines into a single newline
    script = re.sub(r"\n{2,}", "\n", script)

    # Strip leading/trailing whitespace from each line
    lines = [line.strip() for line in script.splitlines()]
    script = " ".join(line for line in lines if line)

    # Lowercase for uniformity
    script = script.lower()

    # Remove extra internal whitespace
    script = re.sub(r"\s+", " ", script).strip()

    return script

In [32]:
df["clean_script"] = df["movie_script"].apply(preprocess_script)

In [ ]:
def segment_sentences(text: str) -> list[str]:
    sentences = nltk.sent_tokenize(text)
    # Filter out very short fragments (< 10 characters) that add no signal
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 10]
    return sentences

In [34]:
# print(type(df["sentences"].iloc[0]))   # is it a list or a string?
# print(df["sentences"].iloc[0])

In [35]:
df["sentences"] = df["clean_script"].apply(segment_sentences)

In [36]:
for s in df["sentences"][0]:
    print(f"{s}: {len(s)}")

midnight express screenplay by oliver stone based on the autobiography by billy hayes with william hoffer revised draft june, 1977 prologue black screen - superimpose: the following is based on a true story.: 207
it october 6, 1970 istanbul, turkey - sound under, sharp: crackle - rip - snip... fade in: a set of clothes on a hotel room bed -- trenchcoat, bulky white turtle-neck sweater, t-shirt, jeans, western style boots.: 212
sounds continue, accentuated.: 29
move across open travel bags on the bed.: 40
clothes, possessions.: 21
continue across furniture, washbasin, toilet...a large room, high old ceilings and windows suggesting ancient europe & design, a haunting greenish afternoon light.: 163
we move to hands, tight - drawing out a strip of adhesive tape, scissors move in tight...snip!: 94
underarm, tight.: 16
tape being laid over it.: 24
back of shoulder.: 17
tape going on.: 14
bellybutton, tight.: 19
tape going then: a harsh rip!: 29
sound and the tape comes off the bellybutton.: 

In [37]:
from itertools import chain

def build_embeddings_fast(df: pd.DataFrame,
                          sbert_model: SentenceTransformer) -> pd.DataFrame:
    """
    Encode all sentences across all movies in a single batched SBERT call.
    Much faster than calling embed_sentences() per movie via apply().
    """
    all_sentences   = df["sentences"].tolist()
    sentence_counts = [len(s) for s in all_sentences]
    flat_sentences  = list(chain.from_iterable(all_sentences))

    print(f"  Encoding {len(flat_sentences)} sentences across "
          f"{len(df)} movies in one batch ...")

    # Single encode call for everything
    all_embeddings = sbert_model.encode(
        flat_sentences,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Split back per movie
    split_embeddings = np.split(all_embeddings, np.cumsum(sentence_counts)[:-1])

    df["sent_emb"]     = [e.tolist() for e in split_embeddings]
    df["movie_vector"] = [e.mean(axis=0).tolist() for e in split_embeddings]

    return df

In [38]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7208.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
df = build_embeddings_fast(df, sbert_model)

  Encoding 2132910 sentences across 881 movies in one batch ...


Batches: 100%|██████████| 33327/33327 [21:14<00:00, 26.14it/s] 
/var/folders/l6/hf84dcw12ns310x798rtj0gc0000gn/T/ipykernel_81550/3887476570.py:28: RuntimeWarning: Mean of empty slice
  df["movie_vector"] = [e.mean(axis=0).tolist() for e in split_embeddings]
/Users/bao/miniconda3/envs/nlp_xai/lib/python3.11/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [40]:
# Define good vs bad rating
df["rating_label"] = df["averageRating"].apply(lambda x: 1 if x >= 7.5 else 0)

In [41]:
df.dtypes

movie_name           str
movie_script         str
averageRating    float64
clean_script         str
sentences         object
sent_emb          object
movie_vector      object
rating_label       int64
dtype: object

In [83]:
#df.to_csv("results/movies_scripts_ratings.csv", index=False, encoding="utf-8")

## MODELLING

In [43]:
def prepare_matrices(df: pd.DataFrame):
    def parse_vector(v):
        if isinstance(v, np.ndarray):
            return v.astype(np.float32)
        if isinstance(v, list):
            return np.array(v, dtype=np.float32)
        if isinstance(v, str):
            return np.fromstring(v.strip("[]"), dtype=np.float32, sep=" ")
        return np.array(v, dtype=np.float32)

    X = np.vstack([parse_vector(v) for v in df["movie_vector"].values])
    y = df["rating_label"].values.astype(np.int32)
    return X, y

In [44]:
# Find which movies have NaN vectors
nan_mask = df["movie_vector"].apply(lambda x: np.isnan(np.array(x, dtype=np.float32)).any())
print(f"Movies with NaN vectors: {nan_mask.sum()} / {len(df)}")

# Check if they have empty sentences
df["n_sentences"] = df["sentences"].apply(len)
print("\nSentence count for NaN movies:")
print(df[nan_mask]["n_sentences"].value_counts())

# Check if they have empty/null scripts
print("\nEmpty scripts:")
print(df[nan_mask]["clean_script"].apply(lambda x: len(str(x)) if pd.notna(x) else 0).describe())

Movies with NaN vectors: 6 / 881

Sentence count for NaN movies:
n_sentences
0    6
Name: count, dtype: int64

Empty scripts:
count    6.0
mean     0.0
std      0.0
min      0.0
25%      0.0
50%      0.0
75%      0.0
max      0.0
Name: clean_script, dtype: float64


In [45]:
df = df[df["sentences"].apply(len) > 0].reset_index(drop=True)
print(f"Remaining movies: {len(df)}")

Remaining movies: 875


In [46]:
df["movie_vector"][0]

[0.015780247747898102,
 0.03375352546572685,
 -0.01086960919201374,
 -0.00567809771746397,
 -0.0006318546365946531,
 -0.004079750273376703,
 0.06284010410308838,
 -0.026289381086826324,
 0.009501130320131779,
 -0.01586984097957611,
 0.0103115513920784,
 -0.0032239335123449564,
 -0.011119574308395386,
 0.001470686518587172,
 -0.00015319924568757415,
 0.002604170236736536,
 0.014772217720746994,
 0.00679976399987936,
 -0.026141256093978882,
 -0.0003177264879923314,
 0.0009450108627788723,
 0.03708820044994354,
 0.03167217969894409,
 0.01874603144824505,
 -0.01700604520738125,
 0.010175812989473343,
 0.010596667416393757,
 -0.004281215835362673,
 -0.003998962696641684,
 -0.04312904551625252,
 -0.01735292747616768,
 -0.0055221025831997395,
 -0.010407395660877228,
 -0.0010352788958698511,
 -0.008611091412603855,
 -0.007859568111598492,
 0.01938842423260212,
 0.021203050389885902,
 0.015008034184575081,
 -0.0015681940130889416,
 -0.001100617810152471,
 -0.023309210315346718,
 -0.008085337467

In [47]:
X, y = prepare_matrices(df)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
print(f"       Train: {len(X_train)} samples  |  Test: {len(X_test)} samples")

       Train: 700 samples  |  Test: 175 samples


In [49]:
def build_models() -> dict:
    from xgboost import XGBClassifier

    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(
                max_iter     = 1000,
                random_state = 42,
                C            = 1.0,
                solver       = "lbfgs",
                class_weight = "balanced",
            )),
        ]),

        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    XGBClassifier(
                n_estimators     = 300,
                max_depth        = 6,
                learning_rate    = 0.05,
                subsample        = 0.8,
                colsample_bytree = 0.8,
                scale_pos_weight = 1.0,
                eval_metric      = "logloss",
                random_state     = 42,
                n_jobs           = -1,
            )),
        ]),
    }


In [50]:
def train_and_evaluate(models, X_train, X_test, y_train, y_test):

    results       = []
    fitted_models = {}

    X_train = np.array(X_train, dtype=np.float32)
    X_test  = np.array(X_test,  dtype=np.float32)
    y_train = np.array(y_train, dtype=np.int32)
    y_test  = np.array(y_test,  dtype=np.int32)

    assert np.isnan(X_train).sum() == 0, "NaNs in X_train"
    assert np.isnan(X_test).sum()  == 0, "NaNs in X_test"

    neg         = (y_train == 0).sum()
    pos         = (y_train == 1).sum()
    scale_ratio = neg / pos
    print(f"  Class ratio (neg/pos): {scale_ratio:.2f}")

    if "XGBoost" in models:
        models["XGBoost"].named_steps["clf"].set_params(
            scale_pos_weight=scale_ratio
        )

    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE — class distribution: {np.bincount(y_train_bal)}")

    for name, pipeline in models.items():
        X_fit = X_train     if name == "XGBoost" else X_train_bal
        y_fit = y_train     if name == "XGBoost" else y_train_bal

        pipeline.fit(X_fit, y_fit)
        y_pred  = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
        auc = roc_auc_score(y_test, y_proba)

        results.append({
            "Model":            name,
            "Accuracy":         round(acc, 4),
            "F1-Score (macro)": round(f1,  4),
            "ROC-AUC":          round(auc, 4),
        })
        fitted_models[name] = pipeline

        print(f"\n{'='*50}")
        print(f"  {name}")
        print(f"{'='*50}")
        print(f"  Accuracy : {acc:.4f}")
        print(f"  F1 Score : {f1:.4f}")
        print(f"  ROC-AUC  : {auc:.4f}")
        print(classification_report(y_test, y_pred,
                                    target_names=["Bad (0)", "Good (1)"],
                                    zero_division=0))

    results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
    best_name  = results_df.iloc[0]["Model"]
    best_model = fitted_models[best_name]

    return results_df, best_name, best_model, fitted_models

In [51]:
models = build_models()
results_df, best_name, best_model, fitted_models = train_and_evaluate(
    models, X_train, X_test, y_train, y_test)

print("\n  ── Evaluation Summary ──")
print(results_df.to_string(index=False))
print(f"\n  ★ Best model: {best_name}")

  Class ratio (neg/pos): 2.55
  After SMOTE — class distribution: [503 503]

  Logistic Regression
  Accuracy : 0.6000
  F1 Score : 0.5157
  ROC-AUC  : 0.5748
              precision    recall  f1-score   support

     Bad (0)       0.73      0.71      0.72       126
    Good (1)       0.30      0.33      0.31        49

    accuracy                           0.60       175
   macro avg       0.52      0.52      0.52       175
weighted avg       0.61      0.60      0.60       175


  XGBoost
  Accuracy : 0.6914
  F1 Score : 0.5179
  ROC-AUC  : 0.6569
              precision    recall  f1-score   support

     Bad (0)       0.73      0.90      0.81       126
    Good (1)       0.38      0.16      0.23        49

    accuracy                           0.69       175
   macro avg       0.56      0.53      0.52       175
weighted avg       0.63      0.69      0.65       175


  ── Evaluation Summary ──
              Model  Accuracy  F1-Score (macro)  ROC-AUC
            XGBoost    0.6914  

In [52]:
fitted_models["Logistic Regression"]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work

## SHAP EXPLAINABILITY

In [ ]:
def compute_shap_values(model_pipeline,
                        X_train: np.ndarray,
                        X_explain: np.ndarray,
                        model_name: str):

    scaler = model_pipeline.named_steps["scaler"]
    clf = model_pipeline.named_steps["clf"]
    X_train_sc   = scaler.transform(X_train)
    X_explain_sc = scaler.transform(X_explain)

    def predict_class1(x):
        return clf.predict_proba(x)[:, 1]

    if "Logistic Regression" in model_name:
        explainer   = shap.LinearExplainer(clf, X_train_sc)
        shap_values = explainer.shap_values(X_explain_sc)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    elif "XGBoost" in model_name:
        explainer   = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X_explain_sc)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    else:
        background  = shap.kmeans(X_train_sc, min(50, len(X_train_sc)))
        explainer   = shap.KernelExplainer(predict_class1, background)
        shap_values = explainer.shap_values(X_explain_sc, nsamples=100)

    shap_values = np.array(shap_values, dtype=np.float32)
    while shap_values.ndim > 2:
        shap_values = shap_values[..., -1]

    assert shap_values.ndim == 2, \
        f"Expected 2D SHAP values, got shape {shap_values.shape}"
    assert shap_values.shape == (X_explain_sc.shape[0], X_explain_sc.shape[1]), \
        f"SHAP shape {shap_values.shape} does not match X_explain shape {X_explain_sc.shape}"

    print(f"  [{model_name}] SHAP values shape: {shap_values.shape}")
    return shap_values, explainer, X_explain_sc

In [58]:
lr_shap_vals, lr_explainer, lr_X_test_sc = compute_shap_values(
        fitted_models["Logistic Regression"], X_train, X_test, "Logistic Regression")

  [Logistic Regression] SHAP values shape: (175, 384)


In [59]:
xgb_shap_vals, xgb_explainer, xgb_X_test_sc = compute_shap_values(
        fitted_models["XGBoost"], X_train, X_test, "XGBoost")

  [XGBoost] SHAP values shape: (175, 384)


### Explain dim

In [ ]:
def infer_dim_label(text: str) -> str:
    keyword_themes = {
        "emotion/feeling":    ["feel", "emotion", "heart", "love", "fear", "hope", "sad", "happy", "angry", "cry"],
        "action/violence":    ["fight", "shoot", "kill", "attack", "run", "chase", "explode", "punch", "weapon"],
        "dialogue/speech":    ["say", "tell", "ask", "speak", "talk", "voice", "word", "answer", "shout", "whisper"],
        "death/loss":         ["die", "dead", "death", "kill", "murder", "loss", "gone", "funeral", "blood", "body"],
        "family/relations":   ["family", "father", "mother", "son", "daughter", "brother", "sister", "wife", "husband", "child"],
        "time/memory":        ["remember", "past", "memory", "time", "ago", "once", "history", "moment", "year", "before"],
        "place/setting":      ["city", "house", "room", "street", "world", "place", "inside", "outside", "door", "window"],
        "mystery/suspense":   ["secret", "hidden", "dark", "shadow", "unknown", "strange", "danger", "afraid", "threat"],
        "comedy/humor":       ["laugh", "funny", "joke", "smile", "comic", "humour", "absurd", "silly", "ridiculous"],
        "moral/ethics":       ["right", "wrong", "truth", "justice", "good", "evil", "believe", "trust", "lie", "honest"],
        "money/power":        ["money", "power", "rich", "poor", "business", "deal", "pay", "control", "authority"],
        "nature/environment": ["water", "fire", "wind", "earth", "tree", "sky", "light", "dark", "rain", "sun"],
        "identity/self":      ["name", "who", "self", "identity", "belong", "real", "person", "human", "soul", "mind"],
        "journey/adventure":  ["travel", "road", "journey", "escape", "find", "search", "explore", "discover", "mission"],
    }

    scores = {}
    for theme, keywords in keyword_themes.items():
        scores[theme] = sum(1 for kw in keywords if kw in text)

    best_theme = max(scores, key=scores.get)
    if scores[best_theme] == 0:
        return "semantic/contextual"
    return best_theme


def build_combined_dim_label_map(df: pd.DataFrame,
                                  all_shap_values: dict,
                                  top_k_dims: int = 50,
                                  top_k_sentences: int = 3) -> dict:
    #collect top dims from every model
    all_important_dims = set()
    for name, sv in all_shap_values.items():
        sv       = np.array(sv)
        mean_abs = np.abs(sv).mean(axis=0)
        top_dims = np.argsort(mean_abs)[-top_k_dims:]
        all_important_dims.update(top_dims.tolist())

    print(f"  Total unique dims across all models: {len(all_important_dims)}")

    #collect all sentences and embeddings from dataset
    print("  Collecting all sentences from dataset ...")
    all_sentences = []
    all_embs      = []

    for _, row in df.iterrows():
        sents = row["sentences"]
        embs  = np.array(row["sent_emb"], dtype=np.float32)
        if len(sents) == 0 or len(embs) == 0:
            continue
        all_sentences.extend(sents)
        all_embs.append(embs)

    all_embs = np.vstack(all_embs)
    print(f"  Total sentences collected: {len(all_sentences)}")

    #infer label for each dim
    dim_label_map = {}
    for dim in all_important_dims:
        dim_activations = all_embs[:, dim]
        top_pos_idx     = np.argsort(dim_activations)[-top_k_sentences:][::-1]
        pos_sentences   = [all_sentences[i] for i in top_pos_idx]
        combined_text   = " ".join(pos_sentences).lower()
        label           = infer_dim_label(combined_text)
        dim_label_map[dim] = f"dim_{dim} [{label}]"

    print(f"  Labels generated for {len(dim_label_map)} dims")
    return dim_label_map

## Plot

In [ ]:
def shap_summary_plot(shap_values, X_explain_sc: np.ndarray,
                      model_name: str,
                      dim_label_map: dict = None,
                      save_path: str = "shap_summary.png"):
    import matplotlib.pyplot as plt

    TOP_K = 20

    # Handle multi-output SHAP (class-1 values for binary classification)
    if isinstance(shap_values, list):
        sv = shap_values[1]
    elif hasattr(shap_values, "values"):
        sv = shap_values.values
        if sv.ndim == 3:
            sv = sv[:, :, 1]
    else:
        sv = np.array(shap_values)

    # Find top-k features by mean absolute SHAP value
    mean_abs = np.abs(sv).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-TOP_K:]

    sv_top = sv[:, top_idx]
    Xv_top = X_explain_sc[:, top_idx]

    # Use inferred labels if available, else fall back to dim_N
    if dim_label_map is not None:
        feature_names = [dim_label_map.get(i, f"dim_{i}") for i in top_idx]
    else:
        feature_names = [f"dim_{i}" for i in top_idx]

    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(sv_top, Xv_top,
                      feature_names=feature_names,
                      show=False,
                      plot_size=None)
    plt.title(f"SHAP Summary Plot – {model_name}\n"
              f"(Top {TOP_K} embedding dimensions by mean |SHAP|)",
              fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  [Saved] SHAP summary plot → {save_path}")

In [69]:
all_shap_values = {
    "Logistic Regression": lr_shap_vals,
    "XGBoost":       xgb_shap_vals
}

dim_label_map = build_combined_dim_label_map(
    df              = df,
    all_shap_values = all_shap_values,
    top_k_dims      = 384,
    top_k_sentences = 3
)

  Total unique dims across all models: 384
  Total sentences collected: 2132910
  Labels generated for 384 dims


In [70]:
shap_summary_plot(lr_shap_vals,  lr_X_test_sc,  "Logistic Regression",
                  dim_label_map=dim_label_map,
                  save_path="shap_summary_logistic_regression.png")

shap_summary_plot(xgb_shap_vals,  xgb_X_test_sc,  "XGBoost",
                  dim_label_map=dim_label_map,
                  save_path="shap_summary_xgb.png")

  [Saved] SHAP summary plot → shap_summary_logistic_regression.png
  [Saved] SHAP summary plot → shap_summary_xgb.png


## Additional Visualization

In [79]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrices(fitted_models, X_test, y_test,
                             save_path="confusion_matrices.png"):
    fig, axes = plt.subplots(1, len(fitted_models), figsize=(5 * len(fitted_models), 4))
    if len(fitted_models) == 1:
        axes = [axes]

    for ax, (name, pipeline) in zip(axes, fitted_models.items()):
        y_pred = pipeline.predict(X_test)
        cm     = confusion_matrix(y_test, y_pred)
        disp   = ConfusionMatrixDisplay(cm, display_labels=["Bad (0)", "Good (1)"])
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name, fontsize=11, fontweight="bold")

    fig.suptitle("Confusion Matrices — All Models", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] Confusion matrices → {save_path}")

plot_confusion_matrices(fitted_models, X_test, y_test)

  [Saved] Confusion matrices → confusion_matrices.png


In [80]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(fitted_models, X_test, y_test,
                    save_path="roc_curves.png"):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 6))
    colors  = ["#4575b4", "#d73027", "#1a9850"]

    for (name, pipeline), color in zip(fitted_models.items(), colors):
        y_proba          = pipeline.predict_proba(X_test)[:, 1]
        fpr, tpr, _      = roc_curve(y_test, y_proba)
        roc_auc          = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f"{name}  (AUC = {roc_auc:.4f})")

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate", fontsize=11)
    ax.set_title("ROC Curves — All Models", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10, loc="lower right")
    ax.xaxis.grid(True, linestyle="--", alpha=0.4)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] ROC curves → {save_path}")

plot_roc_curves(fitted_models, X_test, y_test)

  [Saved] ROC curves → roc_curves.png


In [81]:
def plot_rating_distribution(df, save_path="rating_distribution.png"):
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Raw rating distribution
    axes[0].hist(df["averageRating"], bins=30,
                 color="#4575b4", edgecolor="white", alpha=0.85)
    axes[0].axvline(7.0, color="#d73027", linewidth=2,
                    linestyle="--", label="Threshold (7.0)")
    axes[0].set_xlabel("Average Rating", fontsize=11)
    axes[0].set_ylabel("Number of Movies", fontsize=11)
    axes[0].set_title("Rating Distribution", fontsize=12, fontweight="bold")
    axes[0].legend(fontsize=10)
    axes[0].xaxis.grid(True, linestyle="--", alpha=0.4)

    # Class balance
    counts = df["rating_label"].value_counts().sort_index()
    bars   = axes[1].bar(["Bad Movie (0)", "Good Movie (1)"],
                          counts.values,
                          color=["#d73027", "#4575b4"],
                          edgecolor="white", alpha=0.85)
    for bar, val in zip(bars, counts.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 1,
                     f"{val}\n({val/len(df)*100:.1f}%)",
                     ha="center", va="bottom", fontsize=11, fontweight="bold")
    axes[1].set_ylabel("Number of Movies", fontsize=11)
    axes[1].set_title("Class Distribution (rating >= 7 = Good)",
                       fontsize=12, fontweight="bold")
    axes[1].yaxis.grid(True, linestyle="--", alpha=0.4)

    fig.suptitle("Dataset Overview", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] Rating distribution → {save_path}")

plot_rating_distribution(df)

  [Saved] Rating distribution → rating_distribution.png


In [82]:
def plot_sentence_distribution(df, save_path="sentence_distribution.png"):
    import matplotlib.pyplot as plt

    df["n_sentences"] = df["sentences"].apply(len)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Overall distribution
    axes[0].hist(df["n_sentences"], bins=40,
                 color="#4575b4", edgecolor="white", alpha=0.85)
    axes[0].set_xlabel("Number of Sentences per Script", fontsize=11)
    axes[0].set_ylabel("Number of Movies", fontsize=11)
    axes[0].set_title("Sentence Count Distribution", fontsize=12, fontweight="bold")
    axes[0].xaxis.grid(True, linestyle="--", alpha=0.4)

    # By class
    good = df[df["rating_label"] == 1]["n_sentences"]
    bad  = df[df["rating_label"] == 0]["n_sentences"]
    axes[1].hist(good, bins=30, alpha=0.7, color="#4575b4", label="Good Movie (1)")
    axes[1].hist(bad,  bins=30, alpha=0.7, color="#d73027", label="Bad Movie (0)")
    axes[1].set_xlabel("Number of Sentences per Script", fontsize=11)
    axes[1].set_ylabel("Number of Movies", fontsize=11)
    axes[1].set_title("Sentence Count by Class", fontsize=12, fontweight="bold")
    axes[1].legend(fontsize=10)
    axes[1].xaxis.grid(True, linestyle="--", alpha=0.4)

    fig.suptitle("Script Length Analysis", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] Sentence distribution → {save_path}")

plot_sentence_distribution(df)

  [Saved] Sentence distribution → sentence_distribution.png
